# CLT speed race (version 2)

For a chosen population distribution, the notebook simulates $R$ independent sample paths and follows the standardized mean
$Z_n = \sqrt{n}\,(\bar X_n - \mu)/\sigma$ as $n$ grows from 10 to `nmax`. Each frame shows

1. the histogram of the $R$ simulated values of $Z_n$ with the standard normal density; values outside $[-6, 6]$ are left out and their share is printed;
2. $\hat F(x) - \Phi(x)$, where $\hat F$ is the empirical distribution function of the simulated values, and, as a dashed line, the exact $F_n(x) - \Phi(x)$ when it can be computed (Normal, Uniform, Exponential, Laplace, Chi-square, Student-t with df > 2);
3. the race: the simulated distance $\sup_x|\hat F(x) - \Phi(x)|$ against $n$, the exact distance $D_n$, and the mean noise of a simulation with $R$ replications. Below the noise level the simulation cannot show any further convergence.

The project note (`paper/clt_note.pdf`) explains the three analyses behind these panels. For Cauchy, and for Pareto with tail index $\le 2$, Student-t with df $\le 2$ and F with $d_2 \le 4$, the variance is infinite and $Z_n$ is not defined; analysis 3 of the note shows what the sample mean does in that case.

Run the two cells below, choose a distribution and press **Run**. Recording writes PNG frames and a GIF (Pillow) or an MP4 (needs `ffmpeg`; falls back to a GIF) into the chosen folder.

In [ ]:
import io
import sys
from datetime import datetime
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # the repository root, which holds the cltspeed package

import ipywidgets as W
from IPython.display import Image, clear_output, display

from cltspeed import Distribution, V1_DEFAULTS
from cltspeed import race

In [ ]:
dist = W.Dropdown(options=list(V1_DEFAULTS), value="Exponential", description="Distribution:",
                  style={"description_width": "initial"})
params = {name: {key: W.FloatText(value=value, description=key, layout=W.Layout(width="160px"))
                 for key, value in defaults.items()} for name, defaults in V1_DEFAULTS.items()}
pbox = W.HBox()
seed = W.IntText(value=42, description="Seed:", layout=W.Layout(width="160px"))
R = W.IntSlider(value=2000, min=200, max=20000, step=200, description="R:", continuous_update=False)
nmax = W.IntSlider(value=10000, min=200, max=40000, step=200, description="nmax:", continuous_update=False)
steps = W.IntSlider(value=30, min=4, max=60, description="steps:", continuous_update=False)
record = W.Checkbox(value=False, description="Record", indent=False)
fmt = W.Dropdown(options=[("GIF", "gif"), ("MP4 (needs ffmpeg)", "mp4"), ("PNG frames only", "frames")],
                 value="gif", description="Format:")
fps = W.IntSlider(value=8, min=1, max=30, description="FPS:", continuous_update=False)
outdir = W.Text(value="clt_renders", description="Save in:")
run_button = W.Button(description="Run", button_style="success", icon="play")
status = W.HTML()
out = W.Output()


def show_params(*_):
    pbox.children = tuple(params[dist.value].values())


dist.observe(show_params, names="value")
show_params()


def show(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=90)
    with out:
        clear_output(wait=True)
        display(Image(buffer.getvalue()))


def on_run(_):
    try:
        d = Distribution(dist.value, {k: w.value for k, w in params[dist.value].items()})
    except ValueError as error:
        status.value = f"<b style='color:#b00'>{error}</b>"
        return
    if not d.clt_applies():
        status.value = (f"<b style='color:#b00'>{d.label}: the mean or the variance does not exist, so the "
                        "standardized mean is not defined. See analysis 3 of the project note.</b>")
        return
    folder = None
    if record.value:
        folder = Path(outdir.value).expanduser() / f"{d.name}_{datetime.now():%Y%m%d_%H%M%S}"
    status.value = "Running..."
    rows, path = race.run(d, R=R.value, nmax=nmax.value, steps=steps.value, seed=seed.value,
                          outdir=folder, fmt=fmt.value, fps=fps.value, on_frame=show)
    last = rows[-1]
    text = f"Done. At n = {last['n']:,}: simulated distance {last['simulated']:.4f}"
    if last["exact"] is not None:
        text += f", exact D_n = {last['exact']:.3g}"
    if folder is not None:
        text += f". Saved in {path or folder}"
    status.value = text


run_button.on_click(on_run)
display(W.VBox([W.HBox([dist, seed, run_button]), pbox, W.HBox([R, nmax]), steps,
                W.HBox([record, fmt, fps]), outdir, status, out]))

Without widgets, the same race can be run and saved from Python:

```python
rows, path = race.run(Distribution("Exponential"), R=2000, nmax=10_000, steps=30, seed=42,
                      outdir="clt_renders/exponential", fmt="gif")
```